# 🎙️ VREMOVER Server (Colab)

1. Menu **Runtime → Change runtime type → T4 GPU**
2. Isi `REPO` di cell 1 dengan link GitHub project lu
3. Jalanin cell 1, 2, 3 berurutan
4. Copy link `https://....trycloudflare.com` yang muncul, tempel di app VREMOVER di HP

⚠️ Jangan tutup tab ini selama lagi dipakai. Link berubah tiap server dijalanin ulang.

In [ ]:
# 1) Ambil project
REPO = "https://github.com/USERNAME/VREMOVER"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os
%cd /content
!rm -rf VREMOVER
!git clone --depth 1 -b {BRANCH} {REPO} VREMOVER
%cd /content/VREMOVER/server

In [ ]:
# 2) Install dependensi (sekitar 2-4 menit)
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install flask "audio-separator[gpu]"
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
print('Selesai install.')

In [ ]:
# 3) Jalanin server + bikin link publik
import subprocess, time, re, os

os.chdir('/content/VREMOVER/server')
subprocess.Popen(['python', 'app.py'], stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(4)

tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break

print('\n' + '=' * 56)
print('LINK SERVER VREMOVER (tempel di app HP):')
print(url)
print('=' * 56)
print('\nBiarin cell ini tetap jalan. Log server: /content/server.log')

# tahan cell tetap hidup
for line in tun.stdout:
    pass